# BTP: fully automated Kaggle ECG experiments

**One-time before starting:** enable GPU and Internet, attach PTB-XL and optional compatible wheels,
and create/enable the Kaggle secret **GITHUB_TOKEN** with read access to T-arn21/BTP_project_files.
Configure section 1, then **Run All / Save & Run All**. No source input dataset is required.

The embedded standard-library bootstrap retrieves the secret securely, clones/fast-forwards
`dev/unified-ecg-pipeline` into `/kaggle/working/BTP_project_files`, and refuses local changes/divergence.
The token is never placed in URLs, argv, files, saved configuration or logs.

The fresh-process pipeline installs/checks dependencies without a notebook restart, verifies source/CUDA,
automatically discovers PTB-XL, audits patients/labels, runs a tiny real-data GPU smoke **before** full caching,
then caches data and trains/evaluates the five-superclass and 23-subclass models sequentially.
Progress/heartbeats are emitted at least every 30 seconds while waiting. Results/checks/failures are summarized
under `/kaggle/working/btp_outputs/`. Diagnostic/partial runs are not benchmark completions.
The user-reported successful smoke used 1.0.1; set DATASET_VERSION honestly.
No completed real GPU benchmarks are claimed by this notebook preparation.

In [ ]:
import os
from pathlib import Path

RUN_SUPERCLASS = True
RUN_SUBCLASS = True
RUN_SMOKE = True
RUN_METADATA_AUDIT = True
FULL_EPOCHS = 10
SUPERCLASS_EPOCHS = None  # None uses FULL_EPOCHS
SUBCLASS_EPOCHS = None
FULL_BATCH_SIZE = 8
SMOKE_BATCH_SIZE = 2
DIAGNOSTIC_MODE = False
DIAGNOSTIC_BATCHES = 2  # train and validation only, per epoch; no test evaluation

CHECKOUT_DIR = "/kaggle/working/BTP_project_files"
GIT_BRANCH = "dev/unified-ecg-pipeline"
GIT_TIMEOUT_SECONDS = 600
DATASET_ROOT = os.environ.get("PTBXL_DATA_DIR", "")
DATASET_VERSION = os.environ.get("PTBXL_DATASET_VERSION", "1.0.1")  # operator-declared; match YOUR attached input
OUTPUT_ROOT = "/kaggle/working/btp_outputs"
CACHE_DIR = str(Path(OUTPUT_ROOT) / "cache")
NORMALIZATION_MODE = "sampled"  # sampled (fast) or full (exact)
NORMALIZATION_SAMPLES = 1000
NORMALIZATION_SEED = 42
CACHE_SIGNALS = True
NUM_WORKERS = 0  # reliable first-batch path; only increase after profiling
APPLY_FILTER = True
AUGMENT = True
MIXED_PRECISION = True
SEED = 42
LOG_EVERY_BATCHES = 20
MODEL_SETTINGS = dict(embed_dim=384, hidden_dim=384, n_layers=8, d_state=16,
                      d_conv=4, expand=2, dropout=0.45, bidirectional=True, pooling="mean_attention")

INSTALL_BASE = True  # auto-install missing base dependencies, not Torch
WHEEL_DIR = ""  # auto-find attached mamba_ssm/causal_conv1d wheels if empty
ALLOW_SOURCE_BUILD = True  # explicit fallback when working CUDA packages/wheels are unavailable
MAX_JOBS = 2
RESUME_RUN_DIRS = {"superclass": "", "subclass": ""}  # original run dir with last.pth AND best.pth
CONTINUE_OTHER_TASK_ON_FAILURE = True
ACTIVE = any((RUN_SUPERCLASS, RUN_SUBCLASS, RUN_SMOKE, RUN_METADATA_AUDIT))
NEED_GPU = any((RUN_SUPERCLASS, RUN_SUBCLASS, RUN_SMOKE))
SMOKE_FIT_RECORDS = 16
EARLY_STOPPING_PATIENCE = 15

OPTIMIZER_SETTINGS = dict(learning_rate=3e-4, weight_decay=5e-4, warmup_epochs=3, min_lr=1e-6,
                          label_smoothing=0.05, use_pos_weight=True, pos_weight_max=8.0, grad_clip_norm=1.0)

SETTINGS = {name: globals()[name] for name in ['RUN_SUPERCLASS', 'RUN_SUBCLASS', 'RUN_SMOKE', 'RUN_METADATA_AUDIT', 'FULL_EPOCHS', 'SUPERCLASS_EPOCHS', 'SUBCLASS_EPOCHS', 'FULL_BATCH_SIZE', 'SMOKE_BATCH_SIZE', 'SMOKE_FIT_RECORDS', 'DIAGNOSTIC_MODE', 'DIAGNOSTIC_BATCHES', 'CHECKOUT_DIR', 'GIT_BRANCH', 'GIT_TIMEOUT_SECONDS', 'DATASET_ROOT', 'DATASET_VERSION', 'OUTPUT_ROOT', 'CACHE_DIR', 'NORMALIZATION_MODE', 'NORMALIZATION_SAMPLES', 'NORMALIZATION_SEED', 'CACHE_SIGNALS', 'NUM_WORKERS', 'APPLY_FILTER', 'AUGMENT', 'MIXED_PRECISION', 'SEED', 'LOG_EVERY_BATCHES', 'EARLY_STOPPING_PATIENCE', 'MODEL_SETTINGS', 'INSTALL_BASE', 'WHEEL_DIR', 'ALLOW_SOURCE_BUILD', 'MAX_JOBS', 'RESUME_RUN_DIRS', 'CONTINUE_OTHER_TASK_ON_FAILURE', 'OPTIMIZER_SETTINGS']}


## Automated execution
The following cell is self-contained; all scientific/training modules come from the synchronized checkout.
Existing tracked/untracked changes or local-only commits cause a clear refusal without reset/stash/clean.
Checkpoints, summaries and history use atomic replacement; completed epochs and task artifacts survive later errors.
Kaggle may discard working storage when a session is destroyed: retain saved notebook output for future-session resume.

In [ ]:
"""Self-contained standard-library bootstrap, embedded verbatim in the notebook.

The GitHub secret exists only in memory and Git child environments. It never
appears in URLs, argv, configuration, askpass file contents or returned results.
"""
import importlib
import json
import os
from pathlib import Path
import re
import signal
import subprocess
import sys
import tempfile
import time
import urllib.parse
from datetime import datetime, timezone
from uuid import uuid4


def _git(arguments, *, cwd=None, env=None, timeout=600, label="Git"):
    command = ["git", "-c", "credential.helper=", "-c", "core.hooksPath=" + os.devnull,
               "-c", "core.fsmonitor=false", *arguments]
    process = subprocess.Popen(command, cwd=cwd, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                               stderr=subprocess.PIPE, text=True, errors="replace")
    start = time.monotonic()
    try:
        while True:
            try:
                stdout, stderr = process.communicate(timeout=min(30, timeout))
                break
            except subprocess.TimeoutExpired:
                print(f"{label} still running ({time.monotonic() - start:.0f}s)", flush=True)
                if time.monotonic() - start >= timeout:
                    raise RuntimeError(f"{label} timed out; authentication details suppressed") from None
    except BaseException:
        if process.poll() is None:
            process.kill()
            process.communicate()
        raise
    if process.returncode:
        # Never forward authentication-related subprocess output or exception repr.
        raise RuntimeError(f"{label} failed (exit {process.returncode}). Check Internet, repository read permission, "
                           "branch and Kaggle GITHUB_TOKEN secret. Git output suppressed for credential safety.") from None
    return stdout.strip()


def kaggle_token():
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("GITHUB_TOKEN")
        if not isinstance(value, str) or not value.strip():
            raise ValueError("empty secret")
        return value.strip()
    except Exception:
        raise RuntimeError("Cannot retrieve Kaggle secret GITHUB_TOKEN. Add it in Kaggle Secrets and grant this notebook access.") from None


def sync_repository(checkout, *, branch="dev/unified-ecg-pipeline", token_provider=None,
                    repository_url="https://github.com/T-arn21/BTP_project_files.git", timeout=600):
    """Clone/update clean checkouts by fast-forward only; never stash/reset/clean."""
    checkout = Path(checkout).resolve()
    parsed = urllib.parse.urlsplit(repository_url)
    if parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError("Credential-bearing or parameterized Git URLs are forbidden")
    if parsed.scheme and parsed.scheme != "https" and not Path(repository_url).exists():
        raise ValueError("Only HTTPS repositories or explicit local test repositories are supported")
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]*", branch) or ".." in branch:
        raise ValueError("Invalid branch name")
    env = {k: v for k, v in os.environ.items()
           if not k.startswith(("GIT_TRACE", "GIT_CONFIG_")) and k not in {"GIT_CURL_VERBOSE", "GITHUB_TOKEN", "BTP_GITHUB_TOKEN"}}
    env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="never", GIT_CONFIG_NOSYSTEM="1", GIT_CONFIG_GLOBAL=os.devnull)
    exists = checkout.exists() and any(checkout.iterdir())
    if exists:
        top = _git(["rev-parse", "--show-toplevel"], cwd=checkout, env=env, label="Repository safety check")
        if Path(top).resolve() != checkout:
            raise RuntimeError("Checkout path is not its own Git repository; refusing to overwrite existing files")
        if _git(["status", "--porcelain", "--untracked-files=all"], cwd=checkout, env=env):
            raise RuntimeError("Uncommitted tracked/untracked local changes detected; checkout left untouched. Save them before unattended synchronization.")
        remote = _git(["remote", "get-url", "--all", "origin"], cwd=checkout, env=env)
        if remote != repository_url:
            raise RuntimeError("Existing origin differs from the configured repository; refusing synchronization")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    token = (token_provider or kaggle_token)()
    if not isinstance(token, str) or not token:
        raise RuntimeError("A nonempty GitHub credential is required")
    helper = None
    auth_env = None
    try:
        # File contains only environment lookup, never the secret itself.
        with tempfile.NamedTemporaryFile("w", suffix="_btp_askpass.py", delete=False, encoding="utf-8") as stream:
            helper = Path(stream.name)
            stream.write("#!/usr/bin/env python3\nimport os, sys\n"
                         "print('x-access-token' if 'username' in ' '.join(sys.argv[1:]).lower() "
                         "else os.environ['BTP_GITHUB_TOKEN'])\n")
        helper.chmod(0o700)
        auth_env = dict(env, GIT_ASKPASS=str(helper), BTP_GITHUB_TOKEN=token)
        if not exists:
            _git(["clone", "--single-branch", "--branch", branch, repository_url, str(checkout)],
                 env=auth_env, timeout=timeout, label="Git clone")
        else:
            _git(["fetch", "origin", f"+refs/heads/{branch}:refs/remotes/origin/{branch}"],
                 cwd=checkout, env=auth_env, timeout=timeout, label="Git fetch")
            if _git(["status", "--porcelain", "--untracked-files=all"], cwd=checkout, env=env):
                raise RuntimeError("Local changes appeared during fetch; refusing to update the checkout")
            # switch never forces files; Git refuses conflicting local state.
            _git(["switch", branch], cwd=checkout, env=env, label="Branch selection")
            _git(["merge-base", "--is-ancestor", "HEAD", f"origin/{branch}"], cwd=checkout, env=env,
                 label="Fast-forward safety check (local-only/divergent commits are preserved)")
            _git(["merge", "--ff-only", f"origin/{branch}"], cwd=checkout, env=env, label="Fast-forward update")
        head = _git(["rev-parse", "HEAD"], cwd=checkout, env=env)
        remote_head = _git(["rev-parse", f"origin/{branch}"], cwd=checkout, env=env)
        if head != remote_head or not re.fullmatch(r"[a-f0-9]{40,64}", head):
            raise RuntimeError("Synchronized checkout does not match the fetched remote branch")
        return {"checkout": str(checkout), "branch": branch, "commit": head}
    finally:
        if auth_env is not None:
            auth_env.pop("BTP_GITHUB_TOKEN", None)
        token = None
        if helper is not None:
            helper.unlink(missing_ok=True)


def select_import_root(checkout):
    checkout = Path(checkout).resolve()
    for name in list(sys.modules):
        if name == "src" or name.startswith("src."):
            del sys.modules[name]
    sys.path.insert(0, str(checkout))
    importlib.invalidate_caches()
    module = importlib.import_module("src")
    if Path(module.__file__).resolve().parent != checkout / "src":
        raise RuntimeError("Python resolved src outside the synchronized repository")
    return str(Path(module.__file__).resolve())


def launch_pipeline(checkout, config_path, session_dir):
    """Fresh controller/worker interpreters avoid kernel restarts and stale binaries."""
    checkout = Path(checkout).resolve()
    env = {k: v for k, v in os.environ.items() if k not in {"GITHUB_TOKEN", "BTP_GITHUB_TOKEN"}}
    env["PYTHONPATH"] = str(checkout)
    env["PYTHONUNBUFFERED"] = "1"
    env["PIP_NO_INPUT"] = "1"
    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    env["GIT_TERMINAL_PROMPT"] = "0"
    log = Path(session_dir) / "launcher.log"
    # No secret is available to the installer, worker or its logs.
    command = [sys.executable, "-u", "-m", "src.unattended", "--config", str(config_path)]
    process = subprocess.Popen(command, cwd=checkout, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, errors="replace",
                               start_new_session=os.name != "nt")
    import queue
    import threading
    lines = queue.Queue()
    def read():
        for line in process.stdout:
            lines.put(line)
        lines.put(None)
    threading.Thread(target=read, daemon=True).start()
    try:
        with log.open("a", encoding="utf-8", buffering=1) as stream:
            while True:
                try:
                    line = lines.get(timeout=30)
                except queue.Empty:
                    line = "Pipeline process is still running; see current stage in pipeline_status.json\n"
                if line is None:
                    break
                print(line, end="", flush=True)
                stream.write(line)
        return process.wait()
    except BaseException:
        if process.poll() is None:
            if os.name != "nt":
                os.killpg(process.pid, signal.SIGTERM)
            else:
                process.terminate()
        raise


def execute_notebook(settings):
    """Entry point for the notebook: always print a concise safe execution summary."""
    if any("TOKEN" in key.upper() or "SECRET" in key.upper() for key in settings):
        raise ValueError("Secrets must not be passed in execution settings; use Kaggle Secrets only")
    out = Path(settings["OUTPUT_ROOT"]).resolve()
    session = out / "sessions" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
    session.mkdir(parents=True, exist_ok=True)
    config = dict(settings, SESSION_DIR=str(session))
    summary_path = session / "final_summary.json"
    summary = {"status": "failed", "stage": "GitHub synchronization", "completed": {},
               "checks": {}, "errors": {}, "session_dir": str(session)}
    try:
        synchronized = sync_repository(settings["CHECKOUT_DIR"], branch=settings["GIT_BRANCH"],
                                       timeout=settings["GIT_TIMEOUT_SECONDS"])
        config["SOURCE_REVISION"] = synchronized["commit"]
        config["IMPORT_ROOT"] = select_import_root(synchronized["checkout"])
        print(f"Synchronized {synchronized['branch']} @ {synchronized['commit'][:12]}; imports: {config['IMPORT_ROOT']}", flush=True)
        path = session / "execution_config.json"
        path.write_text(json.dumps(config, indent=2), encoding="utf-8")
        summary["checks"]["repository_sync"] = "passed"
        summary["stage"] = "dependency setup / pipeline worker"
        code = launch_pipeline(synchronized["checkout"], path, session)
        if summary_path.is_file():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
        if code != 0:
            summary["status"] = "failed"
            summary["errors"].setdefault("pipeline", "Pipeline process failed; inspect stage status and logs")
        return summary
    except BaseException as exc:
        # All synchronization exceptions are credential-free by construction.
        if summary_path.is_file():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
        summary["status"] = "failed"
        summary["errors"][summary.get("stage", "launcher")] = str(exc) or type(exc).__name__
        return summary
    finally:
        summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        print("EXECUTION SUMMARY:", json.dumps({"status": summary["status"], "checks": summary.get("checks", {}),
              "completed_experiments": list(summary.get("completed", {})), "errors": summary.get("errors", {}),
              "results": str(session)}, indent=2), flush=True)


if ACTIVE:
    EXECUTION_SUMMARY = execute_notebook(SETTINGS)
    if EXECUTION_SUMMARY["status"] != "complete":
        raise RuntimeError("Unattended pipeline failed; see the printed execution summary and preserved logs")
else:
    EXECUTION_SUMMARY = {"status": "skipped", "checks": {}, "completed": {}, "errors": {}, "session_dir": None}
    print("All run controls disabled; no secret retrieval, network, installation, data loading or GPU work.")


## Execution summary
The worker updates `final_summary.json` after each successful check/task and on failure. If a previous cell
fails, it prints the same concise summary before raising; no later cell is needed to preserve failure evidence.
Full budgets/early stopping use all training records. Diagnostic mode caps train/validation batches and never tests fold 10.

In [ ]:
print("Status:", EXECUTION_SUMMARY["status"])
print("Checks:", EXECUTION_SUMMARY.get("checks", {}))
print("Completed experiments:", list(EXECUTION_SUMMARY.get("completed", {})))
print("Experiments:", EXECUTION_SUMMARY.get("experiments", {}))
print("Errors:", EXECUTION_SUMMARY.get("errors", {}))
print("Results:", EXECUTION_SUMMARY.get("session_dir"))